In [ ]:
from pathlib import Path
import sys

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    return start

repo_root = _find_repo_root(Path.cwd())
workspace_root = repo_root.parent
candidate_src_dirs = [
    repo_root / "src",
    workspace_root / "abstractgraph" / "src",
    workspace_root / "abstractgraph-ml" / "src",
    workspace_root / "abstractgraph-generative" / "src",
]
for src_dir in candidate_src_dirs:
    if src_dir.exists():
        src_str = str(src_dir)
        if src_str not in sys.path:
            sys.path.insert(0, src_str)


# Interpolate Graph Demo
Load PubChem assay graphs and render a small sample.


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import warnings

In [ ]:
# EDS-only plot from predictive_performances_list
import numpy as np
import matplotlib.pyplot as plt

import numpy as np

def expected_gain_weights(ns, eds_params, beta=1.0, eps=1e-8, drop_q=None):
    a, b, alpha = eds_params
    ns = np.asarray(ns, float)
    f = lambda x: a + b * np.power(x, -alpha)
    delta = np.abs(f(2.0*ns) - f(ns))          # |Δ̂_ref(n)|
    if drop_q is not None and 0 < drop_q < 1:
        thr = np.nanquantile(delta, drop_q)
        delta = np.where(delta > thr, delta, 0.0)
    w = np.power(delta, beta)
    w_sum = np.sum(w) + eps
    return w / w_sum

def expected_gain_weighted_eds(ns, r, r_std=None, eds_params=None, beta=1.0, drop_q=None):
    ns = np.asarray(ns, float)
    r = np.asarray(r, float)
    w = expected_gain_weights(ns, eds_params, beta=beta, drop_q=drop_q)
    score = float(np.nansum(w * r))
    score_std = None
    if r_std is not None:
        r_std = np.asarray(r_std, float)
        score_std = float(np.sqrt(np.nansum((w * np.nan_to_num(r_std, nan=0.0))**2)))
    return score, score_std, w


def _fit_learning_curve_from_predictives(ns, y_real, y_ref, s_real=None, s_ref=None, alphas=None):
    ns = np.asarray(ns, float)
    n_all = np.concatenate([ns, 2.0*ns])
    y_all = np.concatenate([y_real, y_ref])
    if s_real is not None and s_ref is not None:
        w_all = 1.0 / np.maximum(np.concatenate([s_real, s_ref])**2, 1e-8)
    else:
        w_all = None
    if alphas is None:
        alphas = np.linspace(0.1, 2.0, 191)
    best = (np.nan, np.nan, 1.0); best_sse = None
    for aexp in alphas:
        x = n_all**(-aexp)
        X = np.stack([np.ones_like(x), x], axis=1)
        if w_all is not None:
            sw = np.sqrt(np.maximum(w_all, 1e-12))
            Xw = X*sw[:,None]; yw = y_all*sw
        else:
            Xw = X; yw = y_all
        try:
            coeff, *_ = np.linalg.lstsq(Xw, yw, rcond=None)
        except Exception:
            continue
        a0, b0 = coeff
        resid = y_all - (a0 + b0*x)
        sse = float(np.sum((resid**2)*(w_all if w_all is not None else 1.0)))
        if best_sse is None or sse < best_sse:
            best_sse = sse; best = (float(a0), float(b0), float(aexp))
    return best

def compute_eds(ns, predictive_performances_list, predictive_performances_std_list=None):
    ns = np.asarray(ns, float)
    arr = np.asarray(predictive_performances_list, float)
    y_real = arr[:,0]; y_gen = arr[:,3]; y_ref = arr[:,2]
    s_real = s_gen = s_ref = None
    if predictive_performances_std_list is not None:
        sarr = np.asarray(predictive_performances_std_list, float)
        s_real = sarr[:,0]; s_gen = sarr[:,3]; s_ref = sarr[:,2]
    a0, b0, alpha = _fit_learning_curve_from_predictives(ns, y_real, y_ref, s_real, s_ref)
    dalpha = max(alpha, 1e-8)
    delta = np.maximum((a0 - y_gen) if b0 < 0 else (y_gen - a0), 1e-8)
    n_eq = (abs(b0)/delta)**(1.0/dalpha)
    r = (n_eq - ns)/np.maximum(ns,1e-8)
    r_std = None
    if s_gen is not None:
        dn_dy = (1.0/dalpha)*(abs(b0)**(1.0/dalpha))*(delta**(-(1.0/dalpha+1.0)))
        n_std = np.abs(dn_dy)*s_gen
        r_std = n_std/np.maximum(ns,1e-8)
    return r, r_std, (a0,b0,alpha)

def plot_eds(ns, r, r_std, params):
    ns = np.asarray(ns,float)
    order = np.argsort(ns)
    ns = ns[order]; r = np.asarray(r)[order]
    r_std = None if r_std is None else np.asarray(r_std)[order]
    a0,b0,alpha = params
    fig, ax = plt.subplots(1,1, figsize=(7.5,4), constrained_layout=True)
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, label='≈1 baseline')
    r_plot = np.where(np.isfinite(r), np.clip(r, -3.0, 3.0), np.nan)
    ax.plot(ns, r_plot, marker='s', linestyle='-', color='black', label='EDS efficiency')
    if r_std is not None:
        cap = np.nanquantile(np.abs(r),0.9) if np.any(np.isfinite(r)) else 1.0
        cap = float(max(cap,0.2))
        r_std_plot = np.where(np.isfinite(r_std), np.minimum(r_std, cap), np.nan)
        ax.fill_between(ns, r_plot - r_std_plot, r_plot + r_std_plot, color='black', alpha=0.12)
    for xi, yi, yraw in zip(ns, r_plot, r):
        if np.isfinite(yi):
            label = f"{yi:.2f}" if np.isfinite(yraw) and abs(yraw) <= 3 else ('>3' if np.isfinite(yraw) and yraw>3 else '<-3')
            ax.annotate(label, (xi, yi), textcoords='offset points', xytext=(6,6), ha='left', va='bottom', color='black', fontsize=8)
    ax.set_xlabel('sample size')
    ax.set_ylabel('ratio')
    ax.grid(True, ls=':')
    ax.set_title(f'Equivalent Data Size (EDS) efficiency (α={alpha:.2f})')
    ax.legend(loc='best')
    print('Equivalent Data Size (EDS) efficiency by size:')
    for n,val in zip(ns, r):
        try: print(f'  n={int(n):4d}  EDS={val:.3f}')
        except Exception: print(f'  n={n}  EDS={val}')
    print(f'Fitted baseline params: a={a0:.4f}, b={b0:.4f}, alpha={alpha:.3f}')
    return fig, (ax,)

import numpy as np
def _sample_graphs(graphs, targets, n, seed=42, replace=False):
    if n is None:
        return graphs, targets
    size = len(graphs)
    if n >= size and not replace:
        n = size
        replace = False
    rng = np.random.default_rng(seed)
    idx = rng.choice(size, size=n, replace=replace)
    graphs_s = [graphs[i] for i in idx]
    targets_s = [targets[i] for i in idx]
    return graphs_s, targets_s

import matplotlib.pyplot as plt
from abstractgraph_generative.generative_performance import concrete_graph_discriminative_generative_quality_score, print_score
from NSPPK.nsppk import NSPPK 

def score_and_visualize_generative_quality(
    generated_graphs, generated_targets,
    train_graphs, train_targets,
    reference_graphs, reference_targets,
    test_graphs, test_targets,
    n_iterations=30,
    verbose=0,
    plot_results=True,
    parallel=True,
    fracional_size=[1,2,3,4,5,7,10,15],
):
    vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)

    print(f'#generated graphs: {len(generated_graphs)}   #real train graphs: {len(train_graphs)}   #real reference graphs: {len(reference_graphs)}   #test graphs: {len(test_graphs)}')
    sizes = [int(len(train_graphs) / frac) for frac in fracional_size][::-1]
    qualities, utilities, indistinguishabilities, similarities = [], [], [], []
    qualities_std, utilities_std, indistinguishabilities_std, similarities_std = [], [], [], []
    exchangeabilities, creativities = [], []
    exchangeabilities_std, creativities_std = [], []
    predictive_performances_list = []
    predictive_performances_std_list = []
    for i, size in enumerate(sizes):
        n_gen_sample = n_train_sample = n_ref_sample = size
        gen_graphs_s, gen_targets_s = _sample_graphs(generated_graphs, generated_targets, n_gen_sample, seed=101)
        train_graphs_s, train_targets_s = _sample_graphs(train_graphs, train_targets, n_train_sample, seed=202)
        reference_graphs_s, reference_targets_s = _sample_graphs(reference_graphs, reference_targets, n_ref_sample, seed=303)
        print(f'[{i+1:2d}/{len(sizes)}]   {size} samples')
        perf = concrete_graph_discriminative_generative_quality_score(
            vectorizer, gen_graphs_s, gen_targets_s,
            train_graphs_s, train_targets_s,
            reference_graphs_s, reference_targets_s,
            test_graphs, test_targets,
            n_iterations=n_iterations, verbose=verbose, parallel=parallel
        )
        if plot_results: plot_generated_summary(perf, title='Generated Quality Summary for sample size '+str(size))
        exchangeability_score, creativity_score, scores, predictive_performances, exchangeability_score_std, creativity_score_std, scores_std, predictive_performances_std = perf
        quality, utility, indistinguishability, similarity = scores
        q_std, u_std, ind_std, sim_std = scores_std
        predictive_performances_list.append(predictive_performances)
        predictive_performances_std_list.append(predictive_performances_std)
        qualities.append(quality); utilities.append(utility); indistinguishabilities.append(indistinguishability); similarities.append(similarity)
        qualities_std.append(q_std); utilities_std.append(u_std); indistinguishabilities_std.append(ind_std); similarities_std.append(sim_std)
        exchangeabilities.append(exchangeability_score); creativities.append(creativity_score)
        exchangeabilities_std.append(exchangeability_score_std); creativities_std.append(creativity_score_std)

    # Compute EDS from predictive performances for plotting and output
    r, r_std, eds_params = compute_eds(sizes, predictive_performances_list, predictive_performances_std_list)
    if plot_results:
        # Plot size sweep bands with optional EDS panel (backward compatible)
        try:
            _ = plot_size_sweep_bands(
                sizes,
                qualities, qualities_std,
                utilities, utilities_std,
                indistinguishabilities, indistinguishabilities_std,
                similarities, similarities_std,
                exchangeabilities, exchangeabilities_std,
                creativities, creativities_std,
                eds_r=r, eds_r_std=r_std, eds_params=eds_params,
            )
        except TypeError:
            # Older plot_size_sweep_bands without EDS args
            _ = plot_size_sweep_bands(
                sizes,
                qualities, qualities_std,
                utilities, utilities_std,
                indistinguishabilities, indistinguishabilities_std,
                similarities, similarities_std,
                exchangeabilities, exchangeabilities_std,
                creativities, creativities_std,
            )
    return (
        sizes,
        qualities, qualities_std,
        utilities, utilities_std,
        indistinguishabilities, indistinguishabilities_std,
        similarities, similarities_std,
        exchangeabilities, exchangeabilities_std,
        creativities, creativities_std,
        predictive_performances_list, predictive_performances_std_list,
        r, r_std, eds_params,
    )   

def expected_gain_weights(ns, eds_params, beta=1.0, eps=1e-8, drop_q=None):
    a, b, alpha = eds_params
    ns = np.asarray(ns, float)
    f = lambda x: a + b * np.power(x, -alpha)
    delta = np.abs(f(2.0*ns) - f(ns))
    if drop_q is not None and 0 < drop_q < 1:
        thr = np.nanquantile(delta, drop_q)
        delta = np.where(delta > thr, delta, 0.0)
    w = np.power(delta, float(beta))
    w_sum = np.sum(w) + eps
    return w / w_sum

def expected_gain_weighted_eds(ns, r, r_std=None, eds_params=None, beta=1.0, drop_q=None):
    ns = np.asarray(ns, float)
    r = np.asarray(r, float)
    w = expected_gain_weights(ns, eds_params, beta=beta, drop_q=drop_q)
    score = float(np.nansum(w * r))
    score_std = None
    if r_std is not None:
        r_std = np.asarray(r_std, float)
        score_std = float(np.sqrt(np.nansum((w * np.nan_to_num(r_std, nan=0.0))**2)))
    return score, score_std, w

# Backward-compatible wrappers with expanded names
def compute_equivalent_data_size(ns, predictive_performances_list, predictive_performances_std_list=None):
    return compute_eds(ns, predictive_performances_list, predictive_performances_std_list)

def plot_equivalent_data_size(ns, r, r_std, params):
    return plot_eds(ns, r, r_std, params)

def expected_gain_weighted_equivalent_data_size(ns, r, r_std=None, eds_params=None, beta=1.0, drop_q=None):
    return expected_gain_weighted_eds(ns, r, r_std=r_std, eds_params=eds_params, beta=beta, drop_q=drop_q)

# Backward-compatibility: old names mapping to Expected-Gain variants
def headroom_weights(ns, eds_params, beta=1.0, eps=1e-8, drop_q=None):
    return expected_gain_weights(ns, eds_params, beta=beta, eps=eps, drop_q=drop_q)

def expected_gain_weighted_eds(ns, r, r_std=None, eds_params=None, beta=1.0, drop_q=None):
    return expected_gain_weighted_eds(ns, r, r_std=r_std, eds_params=eds_params, beta=beta, drop_q=drop_q)


In [ ]:
from abstractgraph.graphs import graph_to_abstract_graph
from abstractgraph.display import display, display_mappings, display_decomposition_graph
def draw(graph, df, nbits=10):
    display_decomposition_graph(df)
    ag = graph_to_abstract_graph(graph, decomposition_function=df, nbits=nbits)
    display(ag, size=(12, 6))
    display_mappings(ag, n_elements_per_row=10)

In [ ]:
from abstractgraph.utils import plot_embedding_2d
def _compute_Z(model, graphs, targets):
    try:
        import umap.umap_ as umap
        from sklearn.neighbors import NeighborhoodComponentsAnalysis
        import numpy as np

        embs = model.transform(graphs)
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message=r".*n_jobs value .* overridden .* by setting random_state.*",
                category=UserWarning,
                module="umap.umap_",
            )
            Z_umap = umap.UMAP(n_components=10, random_state=42, init="random").fit_transform(embs)
        y = np.asarray(targets)
        # Fit NCA only on classes 0 and 1 (exclude class 2), then transform all
        fit_mask = (y == 0) | (y == 1)
        nca = NeighborhoodComponentsAnalysis(n_components=2, random_state=42)
        nca.fit(Z_umap[fit_mask], y[fit_mask])
        Z = nca.transform(Z_umap)
        return Z
    except Exception:
        return None
    
def plot(model, graphs, targets, alpha=.65):
    size = 6
    Z_all = _compute_Z(model, graphs, targets)
    fig, axes = plt.subplots(1, 3, figsize=(3.1*size, size), constrained_layout=True)
    _ = plot_embedding_2d(model, graphs, targets, title_prefix="All data", mode="scatter", alpha=alpha, ax=axes[0], show=False, Z=Z_all)
    _ = plot_embedding_2d(model, graphs, targets, title_prefix="All data", mode="class_union", k=5, alpha=alpha, z=1, ax=axes[1], show=False, show_instances=False, Z=Z_all)
    _ = plot_embedding_2d(model, graphs, targets, title_prefix="All data", mode="class_union", k=11, alpha=alpha, z=1, ax=axes[2], show=False, show_instances=False, Z=Z_all)
    plt.show()

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
from abstractgraph.xml import operator_from_xml_file
import abstractgraph.operators as ops
from abstractgraph.xml import register_from_module
register_from_module(ops)

def list_decomposition_files(directory):
    directory = Path(directory)
    items = []
    for xml_path in sorted(directory.glob("*.xml")):
        name = xml_path.stem
        explanation = None
        reason_synthesis = None
        try:
            root = ET.parse(xml_path).getroot()
            ann = root.find("Annotation")
            if ann is not None and ann.text:
                explanation = ann.text.strip()
            reason = root.find("ReasonSynthesis")
            if reason is not None and reason.text:
                reason_synthesis = reason.text.strip()
        except Exception as exc:
            explanation = f"failed to read: {exc}"
        items.append((name, explanation, reason_synthesis))
    return items

def load_decomposition(name, directory):
    xml_path = Path(directory) / f"{name}.xml"
    return operator_from_xml_file(str(xml_path))

import textwrap
def show_decomposition_files(dir_name):
    decomps = list_decomposition_files(dir_name)
    names = []
    for i, (name, explanation, reason_synthesis) in enumerate(decomps):
        names.append(name)
        print(f"[{i:2d}/{len(decomps)}]   {name} ")
        if explanation:
            print(textwrap.fill(explanation, width=120))
        if reason_synthesis:
            print("Reason synthesis:")
            print(textwrap.fill(reason_synthesis, width=120))
        print("\n")
    return names


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _unpack_perf(perf):
    if len(perf) != 8:
        raise ValueError('perf must be the 8-tuple returned by concrete_graph_discriminative_generative_quality_score')
    return perf

def plot_generated_summary(perf, title='Generated Quality Summary'):
    exchangeability_score, creativity_score, scores, predictive_performances, exchangeability_score_std, creativity_score_std, scores_std, predictive_performances_std = _unpack_perf(perf)
    quality_labels = ['quality', 'utility', 'indistinguishability', 'similarity']
    palette = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00', "#a626b9", "#5fb7ff", "#ffe600"]
    perf_labels = ['real', 'generated', 'real+reference', 'real+generated', 'real_vs_generated']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    axes[0].bar(quality_labels, scores, yerr=scores_std, capsize=4, color=palette[:len(quality_labels)], edgecolor='black', linewidth=0.8)
    axes[0].set_ylim(0, 1.12)
    axes[0].set_title('Quality Metrics')
    axes[1].bar(perf_labels, predictive_performances, yerr=predictive_performances_std, capsize=4, color=palette[:len(perf_labels)], edgecolor='black', linewidth=0.8)
    axes[1].set_ylim(0, 1.12)
    axes[1].tick_params(axis='x', rotation=20)
    for ax in axes:
        ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    axes[1].set_title('Predictive Performance')
    axes[2].bar(['exchangeability', 'creativity'], [exchangeability_score, creativity_score], yerr=[exchangeability_score_std, creativity_score_std], capsize=4, color=palette[:2], edgecolor='black', linewidth=0.8)
    axes[2].set_ylim(0, 1.12)
    axes[2].set_title('Exploitable Scores')
    for ax in axes:
        for bar in ax.patches:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.02,
                f'{height:.2f}',
                ha='center',
                va='bottom',
                fontsize=9,
            )
    fig.suptitle(title)
    plt.show()


# Plot metrics vs sizes: two panels with mean +/- std bands
def plot_size_sweep_bands(
    sizes,
    qualities, qualities_std,
    utilities, utilities_std,
    indistinguishabilities, indistinguishabilities_std,
    similarities, similarities_std,
    exchangeabilities, exchangeabilities_std,
    creativities, creativities_std,
    eds_r=None, eds_r_std=None, eds_params=None,
    palette=None, figsize=(17,5)
):
    import numpy as np, matplotlib.pyplot as plt
    if palette is None:
        palette = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00', '#a626b9', '#5fb7ff', '#ffe600']
    ncols = 3 if eds_r is not None else 2
    # widen fig if EDS included
    fig_w = figsize[0] if ncols==2 else max(figsize[0]*1.4, figsize[0]+7)
    fig, axes = plt.subplots(1, ncols, figsize=(fig_w, figsize[1]), constrained_layout=True)
    if ncols==1:
        axes=[axes]
    def _plot_with_band(ax, x, y, ystd, label, color=None):
        ln, = ax.plot(x, y, marker='o', label=label, color=color)
        c = ln.get_color() if color is None else color
        ax.fill_between(x, np.array(y)-np.array(ystd), np.array(y)+np.array(ystd), color=c, alpha=0.25)
    # Core metrics
    ax = axes[0]
    _plot_with_band(ax, sizes, qualities, qualities_std, 'Quality', color=palette[0])
    _plot_with_band(ax, sizes, utilities, utilities_std, 'Utility', color=palette[1])
    _plot_with_band(ax, sizes, indistinguishabilities, indistinguishabilities_std, 'Indistinguishability', color=palette[2])
    _plot_with_band(ax, sizes, similarities, similarities_std, 'Similarity', color=palette[3])
    ax.set_title('Core Metrics')
    ax.set_xlabel('sample size')
    ax.set_ylabel('score')
    ax.set_ylim(0, 1.025)
    ax.grid(True); ax.legend()
    # Aggregate metrics
    ax = axes[1]
    _plot_with_band(ax, sizes, exchangeabilities, exchangeabilities_std, 'Exchangeability', color=palette[4])
    _plot_with_band(ax, sizes, creativities, creativities_std, 'Creativity', color=palette[5])
    ax.set_title('Aggregate Metrics')
    ax.set_xlabel('sample size')
    ax.set_ylim(0, 1.025)
    ax.grid(True); ax.legend()
    # Optional EDS panel
    if eds_r is not None:
        ax = axes[2]
        eds_r = np.asarray(eds_r)
        eds_r_std = None if eds_r_std is None else np.asarray(eds_r_std)
        r_plot = np.where(np.isfinite(eds_r), np.clip(eds_r, -3.0, 3.0), np.nan)
        ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, label='≈1 baseline')
        ax.plot(sizes, r_plot, marker='s', linestyle='-', color='black', label='EDS efficiency')
        if eds_r_std is not None:
            cap = np.nanquantile(np.abs(eds_r),0.9) if np.any(np.isfinite(eds_r)) else 1.0
            cap = float(max(cap,0.2))
            r_std_plot = np.where(np.isfinite(eds_r_std), np.minimum(eds_r_std, cap), np.nan)
            ax.fill_between(sizes, r_plot - r_std_plot, r_plot + r_std_plot, color='black', alpha=0.12)
        for xi, yi, yraw in zip(sizes, r_plot, eds_r):
            if np.isfinite(yi):
                label = f'{yi:.2f}' if np.isfinite(yraw) and abs(yraw) <= 3 else ('>3' if np.isfinite(yraw) and yraw>3 else '<-3')
                ax.annotate(label, (xi, yi), textcoords='offset points', xytext=(6,6), ha='left', va='bottom', color='black', fontsize=8)
        ax.set_title('Equivalent Data Size (EDS) Efficiency')
        ax.set_xlabel('sample size')
        ax.set_ylabel('ratio')
        ax.grid(True); ax.legend()
    return fig, axes


---

In [ ]:
from coco_grape.data_loader.mol.mol_loader import PubChemLoader
from coco_grape.data_loader.loader import SupervisedDataSetLoader

assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
assay_id = assay_ids[4]
assay_id = '488975' #active 2,634
size = int(2000/.33) #to account for 0.33 for test and 0.5 for reference
print(f"size: {size}")
use_equalized = True

def pubchem_loader():
    return PubChemLoader().load(assay_id)

graphs, targets = SupervisedDataSetLoader(
    pubchem_loader,
    size=size,
    use_equalized=use_equalized,
).load()
targets = np.array(targets)

from abstractgraph.utils import plot_graph_label_counts
_ = plot_graph_label_counts(graphs, top=20, title='Dataset info', log_scale=True)


In [ ]:
from sklearn.model_selection import train_test_split
test_size = 0.33
all_train_graphs, test_graphs, all_train_targets, test_targets = train_test_split(
    graphs,
    targets,
    test_size=test_size,
    stratify=targets if len(np.unique(targets)) > 1 else None,
    random_state=0,
)
reference_split_size = 0.5
train_graphs, reference_graphs, train_targets, reference_targets = train_test_split(
    all_train_graphs,
    all_train_targets,
    test_size=reference_split_size,
    stratify=all_train_targets if len(np.unique(all_train_targets)) > 1 else None,
    random_state=0,
)
print(f"AID {assay_id} graphs: {len(graphs)} (train={len(train_graphs)}, reference={len(reference_graphs)}, test={len(test_graphs)})")

In [ ]:
nbits=14

USE_CTH = False

if USE_CTH:
    dir_name = "cth"
    names = show_decomposition_files(dir_name)

    selection_id = -2

    if names:
        print(f"Total decompositions found: {len(names)}")
        selected_name = names[selection_id]
        print(f"Loading decomposition: {selected_name}")
        decomposition_function = load_decomposition(selected_name, dir_name)
    else:
        print(f"No decompositions found in directory {dir_name}.")
else:
    from abstractgraph.operators import *
    decomposition_function = add(
        neighborhood(radius=(1,2)),
        graphlet(radius=2, number_of_nodes=5),
        cycle(),
        tree(),
    )

draw(graphs[0], decomposition_function, nbits=nbits)

## Generation via interpolation

This loop keeps a pool of candidate graphs, scores them via a predictive model,
and repeatedly samples random pairs 
to interpolate between. For each pair, we generate an interpolation path, score
the intermediates against the chosen target class, keep those above the
acceptance threshold, and add them back into the pool before the next round.
The result is a path-relinking style search that promotes high predicted
activity.


In [ ]:
%%time
from abstractgraph.operators import *
from abstractgraph_ml.feasibility import FeasibilityEstimatorFeatureCannotExist, FeasibilityEstimator, FeasibilityEstimatorNumberOfNodesInRange
from abstractgraph_generative.interpolate import InterpolationEstimator
from abstractgraph_generative.interpolation import InterpolationGenerator
from abstractgraph_ml.estimators import GraphEstimator
from sklearn.ensemble import RandomForestClassifier
from abstractgraph.vectorize import AbstractGraphTransformer

df = neighborhood(radius=1)
fe1 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)

df = cycle()
fe2 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)

df = compose(neighborhood(radius=3), unlabel())
fe3 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)

min_size, max_size = np.quantile([len(g) for g in train_graphs], [0.25, 0.75])
fe4 = FeasibilityEstimatorNumberOfNodesInRange(min_size=min_size, max_size=max_size)

feasibility_estimators = [fe1, fe2, fe3, fe4]
feasibility_estimator = FeasibilityEstimator(feasibility_estimators)

graph_transformer = AbstractGraphTransformer(
    nbits=nbits,
    decomposition_function=decomposition_function,
    return_dense=True,
)

interpolation_estimator = InterpolationEstimator(
    graph_transformer=graph_transformer,
    n_iterations=2,
    n_samples=10,
    k=5,
    feasibility_estimator=feasibility_estimator,
    degree_penalty="auto",
    degree_penalty_mode="multiplicative",
)

generator = InterpolationGenerator(interpolation_estimator).fit(train_graphs, train_targets)

In [ ]:
%%time
from coco_grape.visualizer.mol_display import draw_molecules

generated_graphs, generated_targets = generator.generate(
    n_rounds=2,
    n_pairs=70,
    n_iterations=3,
    best_of=3,
    prob_for_acceptance_interval=(0.57, 0.85),
    verbose=True,
    draw_func=draw_molecules,
)

In [ ]:
from NSPPK.nsppk import NSPPK 
vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)

for _target_id in np.unique(test_targets):
    target_generated_graphs = [g for g, t in zip(generated_graphs, generated_targets) if t == _target_id]
    print(f'#original graphs: {len(test_graphs)}    #generated graphs for target id {_target_id}: {len(target_generated_graphs)}')
    plot(vectorizer,test_graphs + target_generated_graphs,list(test_targets) + [max(test_targets) + 1] * len(target_generated_graphs))

In [ ]:
results = score_and_visualize_generative_quality(
    generated_graphs, generated_targets,
    train_graphs, train_targets,
    reference_graphs, reference_targets,
    test_graphs, test_targets,
    n_iterations=10,
    verbose=0,
    plot_results=True,
    parallel=True,
    fracional_size=[1,2,3,4,5,7,10,15],
)

sizes, qualities, qualities_std, originalities, utilities_std, indistinguishabilities, indistinguishabilities_std, similarities, similarities_std, exchangeabilities, exchangeabilities_std, creativities, creativities_std, predictive_performances_list, predictive_performances_std_list, eds_r, eds_r_std, eds_params = results

# Expected-Gain weighted EDS summary (beta=1.0, drop bottom 10% headroom)
hw_score, hw_std, hw_w = headroom_weighted_eds(sizes, eds_r, eds_r_std, eds_params, beta=1.0, drop_q=0.1)
print(f'Expected-Gain weighted EDS: {hw_score:.3f}' + (f' ± {hw_std:.3f}' if hw_std is not None else ''))


In [ ]:
%%time
from NSPPK.nsppk import NSPPK
from sklearn.ensemble import ExtraTreesClassifier
vec = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)
clf = ExtraTreesClassifier(n_estimators=200, n_jobs=-1, random_state=42)
from abstractgraph_generative.generative_performance import compute_expected_gain_weighted_eds, plot_result
results = compute_expected_gain_weighted_eds(
    generated_graphs, generated_targets,
    train_graphs, train_targets,
    reference_graphs, reference_targets,
    test_graphs, test_targets,
    vec=vec,
    clf=clf,
    fracional_size=(1,2,3,4,5,7,10,15),
    n_repeats=5,
)
print(f'expected_gain_weighted_equivalent_data_size:{results["expected_gain_weighted_equivalent_data_size"]:.3f}')
_ = plot_result(results)


---